In [17]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

In [18]:
def natural_sort_key(value):
    name = value.name if isinstance(value, Path) else str(value)
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r"(\d+)", name)]


PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "L30_data"
MAP_DIR = DATA_DIR / "map-keyframes"
KEYFRAME_EMBEDDING_DIR = DATA_DIR / "keyframe_embeddings"
CAPTION_DIR = DATA_DIR / "captions"
CAPTION_EMBEDDING_DIR = DATA_DIR / "caption_embeddings"

OUTPUT_EMBEDDING = DATA_DIR / "keyframe_embedding.npy"
OUTPUT_METADATA = DATA_DIR / "keyframe_metadata.csv"
OUTPUT_CAPTION_EMBEDDING = DATA_DIR / "caption_embedding.npy"
OUTPUT_CAPTION_METADATA = DATA_DIR / "caption_metadata.csv"

video_ids = sorted(
    {path.stem for path in KEYFRAME_EMBEDDING_DIR.glob("L30_V*.npy")},
    key=natural_sort_key,
)
if not video_ids:
    raise FileNotFoundError(f"No L25 keyframe embeddings found in {KEYFRAME_EMBEDDING_DIR}")

In [19]:
keyframe_embedding_parts = []
keyframe_metadata_parts = []
caption_embedding_parts = []
caption_metadata_parts = []

for video_name in video_ids:
    map_path = MAP_DIR / f"{video_name}.csv"
    keyframe_embedding_path = KEYFRAME_EMBEDDING_DIR / f"{video_name}.npy"
    caption_path = CAPTION_DIR / f"{video_name}.json"
    caption_embedding_path = CAPTION_EMBEDDING_DIR / f"{video_name}.npy"

    for required_path in (map_path, caption_path, caption_embedding_path):
        if not required_path.is_file():
            raise FileNotFoundError(f"Missing input for {video_name}: {required_path}")

    mapping = pd.read_csv(map_path, usecols=["n", "pts_time", "frame_idx", "fps"])
    keyframe_embeddings = np.load(keyframe_embedding_path, allow_pickle=False)
    if keyframe_embeddings.ndim != 2:
        raise ValueError(f"{keyframe_embedding_path.name} must be 2-D")
    if len(mapping) != len(keyframe_embeddings):
        raise ValueError(
            f"Keyframe row mismatch for {video_name}: "
            f"embeddings={len(keyframe_embeddings)}, metadata={len(mapping)}"
        )

    with caption_path.open("r", encoding="utf-8") as file:
        shots = json.load(file)
    if not isinstance(shots, list) or not all(isinstance(shot, dict) for shot in shots):
        raise ValueError(f"Invalid shot metadata format: {caption_path}")

    shots = sorted(shots, key=lambda shot: int(shot["shot_id"]))
    shot_starts = np.asarray([float(shot["start_time"]) for shot in shots])
    shot_ends = np.asarray([float(shot["end_time"]) for shot in shots])
    shot_ids = np.asarray([int(shot["shot_id"]) for shot in shots])

    keyframe_shot_indexes = np.searchsorted(shot_starts, mapping["pts_time"].to_numpy(), side="right") - 1
    if (keyframe_shot_indexes < 0).any() or (keyframe_shot_indexes >= len(shots)).any():
        raise ValueError(f"Some keyframes cannot be assigned to a shot in {video_name}")

    keyframe_metadata = pd.DataFrame(
        {
            "video_name": video_name,
            "n": mapping["n"].astype("int64"),
            "frame_idx": mapping["frame_idx"].astype("int64"),
            "pts_time": mapping["pts_time"].astype("float64"),
            "shot_id": shot_ids[keyframe_shot_indexes].astype("int64")
        }
    )
    keyframe_metadata = keyframe_metadata[
        ["video_name", "n", "frame_idx", "pts_time", "shot_id"]
    ]
    keyframe_embedding_parts.append(keyframe_embeddings.astype(np.float32, copy=False))
    keyframe_metadata_parts.append(keyframe_metadata)

    caption_embeddings = np.load(caption_embedding_path, allow_pickle=False)
    if caption_embeddings.ndim != 2:
        raise ValueError(f"{caption_embedding_path.name} must be 2-D")
    if len(caption_embeddings) != len(shots):
        raise ValueError(
            f"Caption row mismatch for {video_name}: "
            f"embeddings={len(caption_embeddings)}, shots={len(shots)}"
        )

    fps = float(mapping["fps"].iloc[0])
    caption_metadata = pd.DataFrame(
        {
            "video_name": video_name,
            "shot_id": shot_ids,
            "start_frame_idx": np.rint(shot_starts * fps).astype("int64"),
            "end_frame_idx": np.rint(shot_ends * fps).astype("int64"),
            "start_time": shot_starts,
            "end_time": shot_ends,
        }
    )
    caption_metadata_parts.append(caption_metadata)
    caption_embedding_parts.append(caption_embeddings.astype(np.float32, copy=False))

all_keyframe_embeddings = np.concatenate(keyframe_embedding_parts, axis=0)
metadata = pd.concat(keyframe_metadata_parts, ignore_index=True)
all_caption_embeddings = np.concatenate(caption_embedding_parts, axis=0)
caption_metadata = pd.concat(caption_metadata_parts, ignore_index=True)

if len(all_keyframe_embeddings) != len(metadata):
    raise RuntimeError("Final keyframe embedding and metadata row counts differ")
if len(all_caption_embeddings) != len(caption_metadata):
    raise RuntimeError("Final caption embedding and metadata row counts differ")

np.save(OUTPUT_EMBEDDING, all_keyframe_embeddings)
metadata.to_csv(OUTPUT_METADATA, index=False)
np.save(OUTPUT_CAPTION_EMBEDDING, all_caption_embeddings)
caption_metadata.to_csv(OUTPUT_CAPTION_METADATA, index=False)

print(f"Saved {OUTPUT_EMBEDDING}: shape={all_keyframe_embeddings.shape}")
print(f"Saved {OUTPUT_METADATA}: shape={metadata.shape}, columns={list(metadata.columns)}")
print(f"Saved {OUTPUT_CAPTION_EMBEDDING}: shape={all_caption_embeddings.shape}")
print(f"Saved {OUTPUT_CAPTION_METADATA}: shape={caption_metadata.shape}, columns={list(caption_metadata.columns)}")

Saved d:\Truong_pipeline\My_Code\L30_data\keyframe_embedding.npy: shape=(7915, 1152)
Saved d:\Truong_pipeline\My_Code\L30_data\keyframe_metadata.csv: shape=(7915, 5), columns=['video_name', 'n', 'frame_idx', 'pts_time', 'shot_id']
Saved d:\Truong_pipeline\My_Code\L30_data\caption_embedding.npy: shape=(818, 1024)
Saved d:\Truong_pipeline\My_Code\L30_data\caption_metadata.csv: shape=(818, 6), columns=['video_name', 'shot_id', 'start_frame_idx', 'end_frame_idx', 'start_time', 'end_time']
